In [5]:
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
llm = ChatOllama(model="llama3.1", temperature=0)

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

def reasoning_node(state: AgentState):
    """This function acts as the 'brain' node in our graph."""
    messages = state['messages']
    response = llm.invoke(messages)
    return {"messages": [response]}

graph_builder = StateGraph(AgentState)
graph_builder.add_node("reasoning", reasoning_node)
graph_builder.add_edge(START, "reasoning")
graph_builder.add_edge("reasoning", END)
agent = graph_builder.compile()

In [7]:
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage
import json

# 1. Define a strict tool
# the @ is a decorator which and @tool is one that langchain uses. Reads the doc string (triple apostrophe), type hints and packages into 
    # json schema for LLM to understand
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers together. You MUST use this for math."""
    return a * b

tools = [multiply]

# 2. Bind the tool to our LLM so it knows it exists
llm_with_tools = llm.bind_tools(tools)

# 3. Create a Safe Tool Node that catches chaos
def safe_tool_node(state: AgentState):
    """Executes the tool, but catches errors and feeds them back to the LLM."""
    # Get the last message (which should be the LLM's tool call)
    last_message = state['messages'][-1]
    # If the LLM didn't make a tool call, we do nothing and return early
    if not getattr(last_message, 'tool_calls', None):
        return {"messages": []}
    # How does the LLM know to slip the info into the tool calls in the agent state?
        # he LLM just outputs raw text. It is LangChain acting as a translator in the middle that catches that text 
        # and neatly organizes it into the tool_calls section of your state.
        # the bind tools means that LangChain injects a massive system prompt into the LLM before asking your question - 
            # describing it and telling it the type of output it should have if it wants to use the tool
        # LLM realizes it needs the tool based on training, and adheres to that output
        # ChatOllama wrapper in LangChain intercepts the model's raw text response.
        # It leaves the normal .content of the message blank, and instead parses that JSON and neatly stuffs it into an attribute called .tool_calls.
    # Extract the tool call the LLM attempted
    tool_call = last_message.tool_calls[0]
    try:
        # We attempt to run the actual tool function
        # We use a dictionary mapping to find the right tool by name
        available_tools = {"multiply": multiply}
        chosen_tool = available_tools[tool_call["name"]]
        # Execute the tool with the arguments the LLM provided
        result = chosen_tool.invoke(tool_call["args"])
        # If successful, return the result as a ToolMessage
        return {"messages": [ToolMessage(content=str(result), tool_call_id=tool_call["id"])]}
    except Exception as e:
        # THE GUARDRAIL: If the LLM gave us bad JSON or a hallucinated tool name,
        # we catch the crash here. We return the exact error back to the LLM.
        error_msg = f"Error executing tool. Please fix your formatting and try again. Error: {str(e)}"
        print(f"⚠️ Caught an error! Feeding back to LLM: {error_msg}")
        
        return {"messages": [ToolMessage(content=error_msg, tool_call_id=tool_call["id"])]}

print("Step 1 Complete: Safe Tool Node defined with error-catching guardrails!")

Step 1 Complete: Safe Tool Node defined with error-catching guardrails!
